# Train your own small language model

Train a **tiny causal Transformer** from random weights on English
Grimm tales, including *Hansel and Gretel*. Compare generated text before and after training, then save
and reload your model. Expect word fragments and short patterns, not a chatbot.

**Requirements:** Course libraries, a CPU, and internet on the first run.
No API key or GPU needed. Run all cells in order; notebooks 01–06 are not required.

## 1. Setup and text source

In [ ]:
# Locate the project folder.
from pathlib import Path

# Find the repository root from either working directory.
ROOT = Path.cwd()
if not (ROOT / "07_Train_Language_Model").is_dir():
    ROOT = ROOT.parent

In [ ]:
# Import libraries and set up reproducible CPU training.
import copy
import hashlib
import math
import re
import time
import textwrap
from urllib.request import urlopen

# Load plotting and model libraries.
import matplotlib.pyplot as plt
import torch
import tiktoken
from torch import nn
from torch.nn import functional as F

# Small workloads often run faster with fewer CPU threads.
torch.set_num_threads(min(2, torch.get_num_threads()))
# Fix the random seed and use the CPU.
SEED = 42
torch.manual_seed(SEED)
DEVICE = torch.device("cpu")
# Create the folder for the text and trained model.
DATA = ROOT / "data/book_model"
DATA.mkdir(parents=True, exist_ok=True)
print("PyTorch:", torch.__version__, "| Device:", DEVICE)

Source: [*Grimms' Fairy Tales*, Project Gutenberg #2591](https://www.gutenberg.org/ebooks/2591)
([UTF-8 download](https://www.gutenberg.org/ebooks/2591.txt.utf-8)).
The English text credits translations by Edgar Taylor and Marian Edwardes.

We download the collection once and train on multiple complete tales.
Every tenth tale is reserved for validation; *Hansel and Gretel* stays in training.
The local copy retains source and licence notices; the checksum identifies it.
If downloading fails, save the UTF-8 file as `data/book_model/grimm_gutenberg_2591.txt`.

Project Gutenberg lists this edition as public domain in the USA; check local
rules elsewhere.

In [ ]:
# Download the collection and extract its complete tales.
SOURCE_URL = "https://www.gutenberg.org/ebooks/2591.txt.utf-8"
BOOK_PATH = DATA / "grimm_gutenberg_2591.txt"
# Reuse the local download if it already exists.
if not BOOK_PATH.exists():
    with urlopen(SOURCE_URL, timeout=60) as response:
        downloaded = response.read()
    # Save only after the download completes successfully.
    BOOK_PATH.write_bytes(downloaded)

# Identify the source file and normalize line endings.
book_bytes = BOOK_PATH.read_bytes()
source_sha256 = hashlib.sha256(book_bytes).hexdigest()
raw_text = (
    book_bytes.decode("utf-8-sig").replace("\r\n", "\n").replace("\r", "\n")
)


# Find the text between the Gutenberg notices.
def book_body(document):
    """Remove Gutenberg notices from the text used for training."""
    start = re.search(r"\*\*\* START OF .+?\*\*\*", document)
    end = re.search(r"\*\*\* END OF .+?\*\*\*", document)
    if start is None or end is None or start.end() >= end.start():
        raise ValueError("Invalid Gutenberg boundaries: check the text file.")
    return document[start.end() : end.start()]


# Split at tale headings, keeping internal FIRST STORY / SECOND STORY sections.
def extract_tales(document):
    """Exclude the preface, contents, and Gutenberg notices."""
    body = book_body(document)
    first = re.search(r"(?m)^THE GOLDEN BIRD[ \t]*$", body)
    if first is None:
        raise ValueError("First tale heading missing: check the source file.")
    body = body[first.start() :]
    headings = [
        match
        for match in re.finditer(
            r"(?m)^([A-Z][A-Z ,’'!?;:()0-9.\[\]\-]+)[ \t]*$", body
        )
        if match[1] not in {"FIRST STORY", "SECOND STORY"}
    ]
    tales = {}
    for index, heading in enumerate(headings):
        end = (
            headings[index + 1].start()
            if index + 1 < len(headings)
            else len(body)
        )
        title = heading[1].strip()
        if title in tales:
            raise ValueError(f"Duplicate tale heading: {title}")
        tales[title] = body[heading.end() : end].strip()
    return tales


# Hold out entire tales before tokenization or vocabulary selection.
tales = extract_tales(raw_text)
validation_titles = list(tales)[9::10]
training_titles = [title for title in tales if title not in validation_titles]
train_text = "\n\n".join(
    title + "\n" + tales[title] for title in training_titles
)
val_text = "\n\n".join(
    title + "\n" + tales[title] for title in validation_titles
)
assert validation_titles and set(training_titles).isdisjoint(validation_titles)
print(
    "Training tales:",
    len(training_titles),
    "| Validation tales:",
    len(validation_titles),
)
print("Held-out titles:", ", ".join(validation_titles))
print(
    "Training characters:",
    len(train_text),
    "| Validation characters:",
    len(val_text),
)
print("SHA-256:", source_sha256)
print("Local text file:", BOOK_PATH)

## 2. Use word-part tokens

We use the established **`cl100k_base`** BPE tokenizer from `tiktoken`.
Tokens can contain word parts, whole words, spaces, or punctuation.
The first use may download tokenizer data; no model weights are downloaded.

Hold out **every tenth complete tale** before tokenization. To keep our model
small, give only training token IDs compact model IDs. Unseen validation tokens
become `<UNK>`; we report their count. This limited vocabulary is a teaching
shortcut and changes the validation task.

[Tokenizer documentation](https://github.com/openai/tiktoken).

In [ ]:
# Tokenize the separate training and validation tales.

# Load established merge rules; do not train a new tokenizer.
TOKENIZER_NAME = "cl100k_base"
tokenizer = tiktoken.get_encoding(TOKENIZER_NAME)
train_token_ids = tokenizer.encode_ordinary(train_text)

# Reserve model ID zero for unknown tokens; keep only training token IDs.
vocabulary = [None] + sorted(set(train_token_ids))
token_to_model = {
    token_id: model_id
    for model_id, token_id in enumerate(vocabulary)
    if token_id is not None
}


# Convert established tokenizer IDs into compact model IDs.
def encode(string):
    return [
        token_to_model.get(token_id, 0)
        for token_id in tokenizer.encode_ordinary(string)
    ]


# Join token bytes before decoding, since a token may split a Unicode character.
def decode(model_ids):
    pieces = [
        (
            b"<UNK>"
            if model_id == 0
            else tokenizer.decode_single_token_bytes(vocabulary[model_id])
        )
        for model_id in model_ids
    ]
    return b"".join(pieces).decode("utf-8", errors="replace")


# Store compact IDs in tensors for model training and validation.
train_data = torch.tensor(encode(train_text), dtype=torch.long)
val_data = torch.tensor(encode(val_text), dtype=torch.long)
VOCAB_SIZE = len(vocabulary)

# Inspect word parts, vocabulary size, and unknown validation tokens.
example_ids = tokenizer.encode_ordinary("Hansel and Gretel")
print("Word parts:", [tokenizer.decode([i]) for i in example_ids])
print("Model vocabulary:", VOCAB_SIZE, "tokens including <UNK>")
print(
    "Training tokens:", len(train_data), "| Validation tokens:", len(val_data)
)
print(
    "Unknown validation tokens:",
    int((val_data == 0).sum()),
    f"({(val_data == 0).float().mean().item():.1%})",
)
print("Hansel ->", encode("Hansel"), "->", decode(encode("Hansel")))

Targets are the input token IDs shifted by one position.
For `[a, b, c]`, predict `[b, c, d]` using only the preceding tokens.

A batch contains random token windows. `STEPS` counts weight updates, not full
passes through the training tales (epochs).

In [ ]:
# Set training options and create batches of inputs and next-token targets.
CONTEXT = 64
BATCH_SIZE = 32
STEPS = 600  # Maximum updates; early stopping may finish sooner.
EVAL_EVERY = 50
PATIENCE = 3  # Stop after this many checks without a lower validation loss.
LEARNING_RATE = 1e-3


# Sample windows and shift their targets by one token.
def get_batch(data, generator=None):
    starts = torch.randint(
        len(data) - CONTEXT, (BATCH_SIZE,), generator=generator
    )
    # Pair each input window with its next-token targets.
    x = torch.stack([data[i : i + CONTEXT] for i in starts])
    y = torch.stack([data[i + 1 : i + CONTEXT + 1] for i in starts])
    return x.to(DEVICE), y.to(DEVICE)


# Check the text lengths and preview one batch.
assert min(len(train_data), len(val_data)) > CONTEXT
batch_rng = torch.Generator().manual_seed(SEED)
x_demo, y_demo = get_batch(train_data, batch_rng)
print("Input token IDs:", x_demo[0, :10].tolist())
print("Target token IDs:", y_demo[0, :10].tolist())
print("Input:", repr(decode(x_demo[0, :20].tolist())))
print("Targets:", repr(decode(y_demo[0, :20].tolist())))
print("Batch shape:", tuple(x_demo.shape))

## 3. Build the Transformer

**Token IDs → token and position embeddings → two Transformer blocks → scores.**

The **causal mask** prevents attention to future tokens. This makes PyTorch's
`TransformerEncoderLayer` usable for next-token prediction.
`cross_entropy` uses raw scores during training; softmax turns them into
probabilities during generation.

[PyTorch documentation](https://docs.pytorch.org/docs/stable/generated/torch.nn.TransformerEncoderLayer.html).

In [ ]:
# Build a small causal Transformer with randomly initialized weights.
class TinyLanguageModel(nn.Module):
    """Small causal language model with learned position embeddings."""

    # Define the embeddings, attention blocks, and output layer.
    def __init__(self, vocab_size, context=64, width=64, heads=4, layers=2):
        super().__init__()
        self.context = context
        self.token_embedding = nn.Embedding(vocab_size, width)
        self.position_embedding = nn.Embedding(context, width)
        # Initialize each block independently with random weights.
        self.blocks = nn.ModuleList(
            [
                nn.TransformerEncoderLayer(
                    d_model=width,
                    nhead=heads,
                    dim_feedforward=4 * width,
                    dropout=0.1,
                    activation="gelu",
                    batch_first=True,
                    norm_first=True,
                )
                for _ in range(layers)
            ]
        )
        # Normalize hidden vectors and score every possible next token.
        self.norm = nn.LayerNorm(width)
        self.output = nn.Linear(width, vocab_size)

    # Turn input tokens into next-token scores.
    def forward(self, token_ids):
        length = token_ids.shape[1]
        positions = torch.arange(length, device=token_ids.device)
        hidden = self.token_embedding(token_ids) + self.position_embedding(
            positions
        )
        # True means this attention connection is blocked.
        mask = torch.triu(
            torch.ones(
                length, length, dtype=torch.bool, device=token_ids.device
            ),
            diagonal=1,
        )
        # Apply the masked attention blocks in sequence.
        for block in self.blocks:
            hidden = block(hidden, src_mask=mask)
        return self.output(self.norm(hidden))


# Initialize a fresh model with reproducible random weights.
torch.manual_seed(SEED)
config = dict(
    vocab_size=VOCAB_SIZE, context=CONTEXT, width=64, heads=4, layers=2
)
model = TinyLanguageModel(**config).to(DEVICE)
# Report the model size and check its output dimensions.
parameter_count = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {parameter_count:,}")
print("Output shape:", tuple(model(x_demo).shape))

### Generate text before training

Start with `Hansel `, sample the next token, append it, and repeat.
The model sees only the latest **64 tokens**.

**Temperature:** Lower values favour likely tokens; higher values add variety.
Neither guarantees correct text. A fixed seed helps compare outputs.

In [ ]:
# Format output as a clean paragraph with lines of at most 120 characters.
def format_text(text, width=120):
    return textwrap.fill(" ".join(text.split()), width=width)


# Generate text one token at a time before training.
@torch.no_grad()
def generate(
    language_model, prompt="Hansel ", new_tokens=100, temperature=0.8, seed=123
):
    # Validate the prompt and sampling temperature.
    if not prompt or temperature <= 0:
        raise ValueError(
            "The prompt must not be empty; temperature must be > 0."
        )
    language_model.eval()  # Disable dropout.
    # Set up reproducible sampling and encode the prompt.
    rng = torch.Generator(device=DEVICE).manual_seed(seed)
    tokens = torch.tensor([encode(prompt)], dtype=torch.long, device=DEVICE)
    # Predict and append one token at a time.
    for _ in range(new_tokens):
        context = tokens[:, -language_model.context :]
        logits = language_model(context)[:, -1, :] / temperature
        # Unknown is a validation fallback, not a token to generate.
        logits[:, 0] = -torch.inf
        probabilities = F.softmax(logits, dim=-1)
        next_token = torch.multinomial(probabilities, 1, generator=rng)
        tokens = torch.cat([tokens, next_token], dim=1)
    return decode(tokens[0].tolist())


# Keep an untrained sample for the later comparison.
before_text = generate(model)
print(format_text(before_text))

## 4. Learn from prediction errors

**Cross-entropy** measures prediction error; lower is better. Each update:
predict → calculate loss → compute gradients → update weights with AdamW.

Check fixed training and validation windows every 50 updates. **Early stopping**
ends training after three checks without improvement and restores the weights
with the lowest validation loss. This also happens if the 600-update limit is reached.
For comparisons, rerun from model creation to reset the weights.

In [ ]:
# Train the model and track training and validation loss.
@torch.no_grad()
def estimate_loss(language_model, batches=10):
    # Disable dropout while measuring loss.
    was_training = language_model.training
    language_model.eval()
    losses = {}
    # Evaluate fixed random windows from each text section.
    for name, data in [("train", train_data), ("validation", val_data)]:
        rng = torch.Generator().manual_seed(SEED + 1)
        values = []
        for _ in range(batches):
            x, y = get_batch(data, rng)
            logits = language_model(x)
            loss = F.cross_entropy(
                logits.reshape(-1, VOCAB_SIZE), y.reshape(-1)
            )
            values.append(loss.item())
        losses[name] = sum(values) / len(values)
    # Restore the previous mode without changing weights.
    language_model.train(was_training)
    return losses


# Set up the optimizer and an independent batch sampler.
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
training_rng = torch.Generator().manual_seed(SEED + 2)
# Measure loss before the first weight update.
history = [{"step": 0, **estimate_loss(model)}]
# Keep an independent copy of the best weights, including the initial model.
best_loss = history[0]["validation"]
best_step = 0
best_state = copy.deepcopy(model.state_dict())
checks_without_improvement = 0
started = time.perf_counter()
print(f"Uniform distribution: {math.log(VOCAB_SIZE):.3f}")
print("Before training:", history[0])

# Learn from a new batch on each step.
for step in range(1, STEPS + 1):
    model.train()
    x, y = get_batch(train_data, training_rng)
    logits = model(x)
    loss = F.cross_entropy(logits.reshape(-1, VOCAB_SIZE), y.reshape(-1))
    # Clear old gradients, compute new ones, and update the weights.
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()

    # Record training and validation loss at regular intervals.
    if step % EVAL_EVERY == 0 or step == STEPS:
        scores = estimate_loss(model)
        history.append({"step": step, **scores})
        print(
            f"Step {step:4d} | Training {scores['train']:.3f}"
            f" | Validation {scores['validation']:.3f}"
        )

        # Save every new best; stop after several checks without improvement.
        if scores["validation"] < best_loss:
            best_loss = scores["validation"]
            best_step = step
            best_state = copy.deepcopy(model.state_dict())
            checks_without_improvement = 0
        else:
            checks_without_improvement += 1
        if checks_without_improvement >= PATIENCE:
            print(f"Early stopping at step {step}.")
            break

# Generate and save from the best checkpoint, not the final update.
trained_steps = step
model.load_state_dict(best_state)
model.eval()
print(f"Restored step {best_step} | Validation {best_loss:.3f}")

# Report the total training and evaluation time.
elapsed = time.perf_counter() - started
print(f"Training time including evaluation: {elapsed:.1f} seconds")

In [ ]:
# Plot the learning curves against the uniform prediction baseline.
steps = [row["step"] for row in history]
plt.figure(figsize=(8, 4))
# Draw one curve for each text section.
for key, label in [("train", "Training"), ("validation", "Validation")]:
    plt.plot(steps, [row[key] for row in history], marker="o", label=label)
# Add the loss of uniform random predictions as a reference.
plt.axhline(
    math.log(VOCAB_SIZE),
    linestyle="--",
    color="gray",
    label="Uniform distribution",
)
# Mark the checkpoint selected by validation loss.
plt.axvline(best_step, linestyle=":", color="black", label="Best checkpoint")
# Label and display the chart.
plt.xlabel("Weight updates")
plt.ylabel("Cross-entropy per token (natural logarithm)")
plt.legend()
plt.tight_layout()
plt.show()

Falling validation loss indicates better predictions on held-out text.
Falling training loss with rising validation loss suggests **overfitting**;
small fluctuations may reflect sampling noise.
The `<UNK>` fallback means these losses do not measure the full tokenizer vocabulary.

## 5. Compare before and after training

Use the same prompt, temperature, and seed. Look for word fragments and repetition.
The sample after training uses the best validation checkpoint.
A small model and 64 tokens of context still limit coherence.

In [ ]:
# Compare generated text before and after training and at a lower temperature.
after_text = generate(model)

# Show both samples with the same prompt and sampling settings.
print("BEFORE TRAINING")
print(format_text(before_text))
print()
print("AFTER TRAINING")
print(format_text(after_text))
print()

# Try more concentrated sampling with a lower temperature.
print("DIFFERENT TEMPERATURE (0.5)")
print(format_text(generate(model, temperature=0.5)))

## 6. Save and reload your model

Save weights, architecture, tokenizer name, compact ID mapping, and source details to generate text later.
Resuming training exactly also requires optimizer and random-generator states.
Here we reload our own checkpoint and check that the output matches.

In [ ]:
# Save the model, reload it, and verify that its generated text matches.
CHECKPOINT = DATA / "tiny_grimm.pt"
torch.save(
    {
        "state_dict": model.state_dict(),
        "config": config,
        "vocabulary": vocabulary,
        "tokenizer_name": TOKENIZER_NAME,
        "tiktoken_version": tiktoken.__version__,
        "source_url": SOURCE_URL,
        "source_sha256": source_sha256,
        "seed": SEED,
        "steps": trained_steps,
        "max_steps": STEPS,
        "best_step": best_step,
        "best_validation_loss": best_loss,
        "patience": PATIENCE,
        "eval_every": EVAL_EVERY,
        "training_titles": training_titles,
        "validation_titles": validation_titles,
        "torch_version": str(torch.__version__),
        "history": history,
    },
    CHECKPOINT,
)

# Rebuild the architecture and restore its learned weights.
saved = torch.load(CHECKPOINT, map_location=DEVICE, weights_only=True)
restored = TinyLanguageModel(**saved["config"]).to(DEVICE)
restored.load_state_dict(saved["state_dict"])

# The tokenizer must match the checkpoint, including in a new session.
tokenizer = tiktoken.get_encoding(saved["tokenizer_name"])
vocabulary = saved["vocabulary"]
token_to_model = {
    token_id: model_id
    for model_id, token_id in enumerate(vocabulary)
    if token_id is not None
}

# Check that reloading preserves the generated output.
restored_text = generate(restored)
assert restored_text == after_text
print("Saved:", CHECKPOINT)
print("Reloaded: identical text with the same sampling seed.")

## Check your understanding

Answer briefly using the outputs above. Try first, then open the hints.

1. Why are the targets shifted by exactly one token?
2. What would happen during training without a causal mask?
3. Is one training step the same as a complete pass through all training tales?
4. Which step had the lowest validation loss? Did early stopping trigger, and
   why might the selected step differ from the final update?
5. Generate text with temperatures of 0.5, 0.8, and 1.2. What differences do you see?
6. Why can the model not reliably answer questions about the tales, even when
   validation loss decreases?
7. What additional information would you need to save to resume training exactly?

**Your answers:**

1. …
2. …
3. …
4. …
5. …
6. …
7. …

<details>
<summary style="color: #B8860B; font-weight: bold; cursor: pointer;">Hints for self-check</summary>

1. Predict the next unseen token from the preceding context.

2. The model could see the answer in a later input position.

3. No. Each update uses a batch of random, possibly overlapping windows.

4. Check the restored step and learning curves. Three checks without improvement
   stop training; the best weights are restored even at the update limit.

5. Lower temperatures can increase repetition; higher values add variety and errors.

6. Next-token prediction is not question-answer training.

7. Save optimizer state, random-generator states, and training position; keep the same data and environment.

</details>

**Try next:** A larger model vocabulary, wider layers, or a longer context. These need more
computation. Instruction-tuning is a further step towards an assistant.

**Local files:** `data/book_model/grimm_gutenberg_2591.txt` is the download;
`data/book_model/tiny_grimm.pt` is your trained model. Both stay outside Git.
Deleting the checkpoint means training again.